# Lecture 3 — Class Exercise
## Line Charts & Slopegraphs: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
1. No spaghetti — multiple lines must use grey + single highlight
2. Remove clutter: no chart borders, no heavy gridlines, no legend if you can label directly
3. Insight title — states the finding, not the topic
4. Carry forward from Lecture 2: white background, Arial font, professional quality


In [ ]:
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('../data/co2_emissions.csv')

OUTPUT_DIR = os.path.join(os.getcwd(), 'lecture03_output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())
print(f"
Outputs will be saved to: {OUTPUT_DIR}")


In [ ]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


---
## Task 1 — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#DDDDDD), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

> 💡 `df[df['Region'] == 'Asia']` to filter; use `go.Figure()` with a loop for per-country control


In [ ]:
# Task 1 — Multi-series line chart with highlight
# ------------------------------------------------

asia = df[df['Region'] == 'Asia'].copy()
highlight = 'China'
grey = '#DDDDDD'
highlight_color = '#E63946'

fig1 = go.Figure()

# Grey background lines for all non-highlighted countries
for country in asia['Country'].unique():
    if country == highlight:
        continue
    d = asia[asia['Country'] == country]
    fig1.add_trace(go.Scatter(
        x=d['Year'], y=d['CO2_Mt'],
        mode='lines',
        line=dict(color=grey, width=1.5),
        showlegend=False,
        hovertemplate=f'{country}: %{{y:.0f}} Mt<extra></extra>'
    ))

# Highlighted country
d_highlight = asia[asia['Country'] == highlight]
fig1.add_trace(go.Scatter(
    x=d_highlight['Year'], y=d_highlight['CO2_Mt'],
    mode='lines',
    line=dict(color=highlight_color, width=3),
    showlegend=False,
    hovertemplate=f'{highlight}: %{{y:.0f}} Mt<extra></extra>'
))

# Direct label at the end of the highlighted line
last = d_highlight[d_highlight['Year'] == d_highlight['Year'].max()].iloc[0]
fig1.add_annotation(
    x=last['Year'], y=last['CO2_Mt'],
    text=f"<b>{highlight}</b>",
    showarrow=False, xanchor='left', xshift=8,
    font=dict(color=highlight_color, size=13)
)

fig1.update_layout(
    template='simple_white',
    title_text="China's CO2 emissions tripled since 2000, dwarfing every other Asian nation",
    title_x=0.02,
    xaxis_title='',
    yaxis_title='CO2 emissions (Mt)',
    height=480,
    margin=dict(r=80)
)
fig1.update_yaxes(rangemode='tozero')

fig1.write_image(os.path.join(OUTPUT_DIR, 'task1_asia_line_highlight.png'))
print('Saved task1_asia_line_highlight.png')
fig1.show()


---
## Task 2 — Slopegraph: Regional Change 2000 vs 2022

**What to build:** A slopegraph comparing **average regional CO2 emissions** between 2000 and 2022.

**Requirements:**
- One line per region (not per country — aggregate first)
- Colour: regions that increased = one colour; decreased = another
- Values labelled at both ends of each line
- No y-axis tick labels (the endpoint labels make them redundant)
- Insight title stating which regions moved most

> 💡 `df.groupby(['Region','Year'])['CO2_Mt'].mean().reset_index()` then filter to 2000 and 2022


In [ ]:
# Task 2 — Slopegraph: regional averages 2000 vs 2022
# -----------------------------------------------------

slope_data = (
    df[df['Year'].isin([2000, 2022])]
    .groupby(['Region', 'Year'])['CO2_Mt']
    .mean()
    .reset_index()
)

pivot = slope_data.pivot(index='Region', columns='Year', values='CO2_Mt')
pivot['increased'] = pivot[2022] > pivot[2000]

color_up = '#E63946'    # increased
color_down = '#457B9D'  # decreased

fig2 = go.Figure()

for region, row in pivot.iterrows():
    color = color_up if row['increased'] else color_down
    v2000, v2022 = row[2000], row[2022]

    fig2.add_trace(go.Scatter(
        x=[2000, 2022], y=[v2000, v2022],
        mode='lines+markers',
        line=dict(color=color, width=2.5),
        marker=dict(size=8, color=color),
        showlegend=False,
        hovertemplate=f'{region}<extra></extra>'
    ))

    # Left label
    fig2.add_annotation(
        x=2000, y=v2000,
        text=f"{region}  {v2000:.0f} Mt",
        showarrow=False, xanchor='right', xshift=-6,
        font=dict(color=color, size=11)
    )
    # Right label
    fig2.add_annotation(
        x=2022, y=v2022,
        text=f"{v2022:.0f} Mt  {region}",
        showarrow=False, xanchor='left', xshift=6,
        font=dict(color=color, size=11)
    )

fig2.update_layout(
    template='simple_white',
    title_text='Asia surged while North America and Europe cut CO2 — a widening regional divide since 2000',
    title_x=0.02,
    height=520,
    xaxis=dict(tickvals=[2000, 2022], ticktext=['2000', '2022'], showgrid=False),
    yaxis=dict(showticklabels=False, showgrid=False, zeroline=False),
    margin=dict(l=160, r=160)
)

fig2.write_image(os.path.join(OUTPUT_DIR, 'task2_slopegraph_regions.png'))
print('Saved task2_slopegraph_regions.png')
fig2.show()
